# Task 06 — Offline Candidate Union and RRF

This notebook is a read-only analytical view of the immutable task-06 candidate datasets and RRF artifact. It never fits or invokes candidate models; source generation and ensemble evaluation remain physically separated.

In [ ]:
import json
from pathlib import Path

import polars as pl

from candidate_pipeline import CandidateUnionConfig, cross_score_columns, generator_columns
from ensemble import RRFEnsembleModel
from interfaces import FINAL_RECOMMENDATION_SCHEMA
from validation import validate_final_recommendations

dataset_dir = Path("artifacts/task06_candidate_datasets_v1")
ensemble_dir = Path("artifacts/task06_candidate_ensemble_v1")
dataset_config = json.loads((dataset_dir / "config.json").read_text(encoding="utf-8"))
dataset_metrics = json.loads((dataset_dir / "metrics.json").read_text(encoding="utf-8"))
ensemble_config = json.loads((ensemble_dir / "config.json").read_text(encoding="utf-8"))
ensemble_metrics = json.loads((ensemble_dir / "metrics.json").read_text(encoding="utf-8"))
canonical_metrics = json.loads((ensemble_dir / "canonical_metrics.json").read_text(encoding="utf-8"))
recommendations = pl.read_parquet(ensemble_dir / "recommendations.parquet")
model = RRFEnsembleModel.from_artifact(ensemble_dir / "model")

In [ ]:
stage_winners = pl.DataFrame(
    [
        {
            "stage": stage["stage"],
            "winner": stage["winner"],
            **stage["summary"][stage["winner"]],
        }
        for stage in ensemble_metrics["selection_stages"]
    ]
).select(
    "stage",
    "winner",
    "mean_precision_at_20_all_targets",
    "mean_precision_at_20_labeled_users",
    "mean_candidate_oracle_p20_all_targets",
    "mean_candidate_recall",
    "mean_mean_candidate_count",
)
display(stage_winners)

In [ ]:
canonical_summary = pl.DataFrame(
    [
        {
            "selected_config": ensemble_metrics["selected_config_id"],
            "precision_at_20_all_targets": canonical_metrics["precision_at_20_all_targets"],
            "precision_at_20_labeled_users": canonical_metrics["precision_at_20_labeled_users"],
            "candidate_recall": canonical_metrics["candidate_recall"],
            "candidate_oracle_p20_all_targets": canonical_metrics["candidate_oracle_p20_all_targets"],
            "oracle_minus_rrf_p20_all_targets": canonical_metrics["oracle_minus_rrf_p20_all_targets"],
            "mean_candidate_count": canonical_metrics["mean_candidate_count"],
        }
    ]
)
source_hits = pl.DataFrame(
    [
        {
            "source": source,
            "relevant_hits": hits,
            "exclusive_hits": canonical_metrics["exclusive_relevant_hits"][source],
        }
        for source, hits in canonical_metrics["source_relevant_hits"].items()
    ]
).sort("source")
display(canonical_summary)
display(source_hits)

In [ ]:
source_order = dataset_config["materialized_union"].get(
    "source_order",
    ["global_popularity", "recency_popularity", "item2item", "implicit_als"],
)
materialized = CandidateUnionConfig.from_mapping(
    {name: dataset_config["materialized_union"]["source_caps"][name] for name in source_order},
    total_cap=dataset_config["materialized_union"]["total_cap"],
)
canonical_manifest = json.loads((dataset_dir / "folds/canonical/dataset_manifest.json").read_text(encoding="utf-8"))
parts = sorted((dataset_dir / "folds/canonical/union_features").glob("part-*.parquet"))
union_schema = pl.scan_parquet([path.as_posix() for path in parts]).collect_schema()
feature_contract = pl.DataFrame(
    [
        {
            "source": source,
            "generated_column": generator_columns(source)["generated"],
            "generator_rank_column": generator_columns(source)["rank"],
            "cross_score_column": cross_score_columns(source)["score"],
            "cross_score_available_column": cross_score_columns(source)["available"],
            "cross_score_availability": canonical_manifest["metrics"]["cross_score_availability"][source],
        }
        for source in materialized.source_names
    ]
)
assert all(column in union_schema for column in feature_contract["cross_score_column"])
assert all(column in union_schema for column in feature_contract["generated_column"])
display(feature_contract)

In [ ]:
assert dataset_config["kind"] == "task06_offline_candidate_datasets"
assert ensemble_config["kind"] == "task06_rrf_ensemble"
assert ensemble_config["selection"]["canonical_isolation"]
assert model.config.to_dict() == ensemble_metrics["selected_config"]
assert recommendations.schema == FINAL_RECOMMENDATION_SCHEMA
validate_final_recommendations(recommendations, expected_k=20)
artifact_checks = {
    "offline_folds": list(dataset_metrics["folds"]),
    "canonical_union_rows": canonical_manifest["rows"],
    "canonical_union_parts": canonical_manifest["part_count"],
    "recommendation_users": recommendations.height,
    "selected_config_id": model.config.config_id,
    "source_order": list(model.config.candidate_config.source_names),
}
artifact_checks

## Interpretation

Task 06 freezes each candidate source before ensemble selection. Generator membership answers why a pair entered the union, while cross-model scores provide independent evidence for every scoreable pair and are reserved for the later CatBoost dataset. RRF uses only generator ranks as a transparent baseline. The gap between candidate oracle P@20 and RRF P@20 is the headroom available to the supervised ranker.